Input batch Excel files
(Structured_Philippine_Addresses_Unmatched_part_XXX.xlsx)
     │
     ▼
Cell 0  Environment check
     │
     ▼
Cell 1  Configuration
(paths, thresholds, batch range, USE_API=False)
     │
     ▼
Cell 3  Load reference tables
(dim_location + alias CSV)
     │
     ▼
Cell 4  Stage 1 — Alias normalization
(single-pass compiled regex + duplicate-token cleanup)
     │
     ▼
Cell 6  Stage 2/3 — Strict evidence-based resolver
(explicit city / explicit unique barangay / curated area cues)
     │
     ├─ valid evidence found ───────────────► city/province/region (+ optional barangay) assigned
     │
     └─ missing/conflicting evidence ───────► invalid_reason set (no-assumption policy)
     │
     ▼
Cell 7  Run over all rows (batch loop + tqdm)
     │
     ▼
Cell 8  API stage skipped (forced no-op)
(USE_API=False; no external geocoding calls)
     │
     ▼
Cell 9  Merge + final validity flag
(is_valid = city_required_pass)
     │
     ▼
Cell 10 Export outputs
 ├─ verified_addresses_parts_XXX_YYY.xlsx
 ├─ invalid_addresses_parts_XXX_YYY.xlsx
 └─ combined_addresses_parts_XXX_YYY.xlsx (full diagnostics)
     │
     ▼
Cells 11+ Summary + diagnostics
(confidence distribution, invalid patterns, unresolved probes)


## Cell 0 — Environment check

Verifies all required Python packages (pandas, numpy, rapidfuzz, tqdm, openpyxl, xlsxwriter) are installed.

If any packages are missing, a note will be printed with installation instructions. This must pass before the pipeline can run.

In [1]:
import importlib, sys

REQUIRED = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'rapidfuzz': 'rapidfuzz',
    'tqdm': 'tqdm',
    'openpyxl': 'openpyxl',
    'xlsxwriter': 'xlsxwriter',
}

missing = []
for pkg_label, pkg in REQUIRED.items():
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'n/a')
        print(f'  ✅  {pkg_label:<14} {ver}')
    except ImportError:
        print(f'  ❌  {pkg_label:<14} NOT FOUND')
        missing.append(pkg)

if missing:
    print(f'\n⚠️  Install missing packages:')
    print(f'    pip install {" ".join(missing)}')
else:
    print('\n✅  All dependencies satisfied — ready to run.')


  ✅  pandas         3.0.1
  ✅  numpy          2.4.3
  ✅  rapidfuzz      3.14.3
  ✅  tqdm           4.67.3
  ✅  openpyxl       3.1.5
  ✅  xlsxwriter     3.2.9

✅  All dependencies satisfied — ready to run.


## Cell 1 — Configuration

Edit the paths and thresholds here. Everything else in the pipeline reads from these variables.

| Variable | Default | Meaning |
|---|---|---|
| `CITY_SCORE_HIGH` | 88 | Auto-accept without API call |
| `CITY_SCORE_LOW` | 65 | Minimum to even consider a city match |
| `PROV_SCORE_HIGH` | 88 | Auto-accept province |
| `PROV_SCORE_LOW` | 60 | Minimum province score |
| `USE_API` | `True` | Set `False` for fuzzy-only mode (faster, less accurate) |
| `API_QUERY_MODE` | `'aggressive'` | API search mode: `'strict'` = minimal queries, `'aggressive'` = typo-tolerant fallbacks |
| `MAX_ROWS` | `None` | Set an integer (e.g. `100`) to process only a slice for testing |


In [2]:
from pathlib import Path

# -- File paths --
BASE_DIR = Path("..") / ".." / "data"   # notebook is in update_address/notebooks

# Single-file input mode
INPUT_FILE   = str(BASE_DIR / "sample"  / "ps_address_mapping_0401.csv")
DIM_LOCATION = str(BASE_DIR / "mapping" / "dim_location_20260316_v2.csv")
ALIAS_FILE   = str(BASE_DIR / "utils"   / "ph_address_alias_extended_v4.csv")
OUT_VERIFIED = str(BASE_DIR / "output"  / "verified_addresses.xlsx")
OUT_INVALID  = str(BASE_DIR / "output"  / "invalid_addresses.xlsx")

# -- Batch input disabled (single-file run) ------------------------------------
input_paths = []

# Example batch mode template (enable only when needed):
# input_paths = [
#     Path('../../data/sample/Structured_Philippine_Addresses_Unmatched/') / f'Structured_Philippine_Addresses_Unmatched_part_{i:03d}.xlsx'
#     for i in range(1, 51)
# ]

# -- Fuzzy-match thresholds (0-100) ------------------------------------------
CITY_SCORE_HIGH  = 88
CITY_SCORE_LOW   = 65
PROV_SCORE_HIGH  = 88
PROV_SCORE_LOW   = 60
BRGY_SCORE_MIN   = 75

# -- API settings kept for optional future use (currently disabled) -----------
PHOTON_URL = 'https://photon.komoot.io/api'
PHOTON_UA  = 'ph-address-pipeline/1.0 (research)'
BATCH_DELAY = 1.1
MAX_RETRIES = 1

# -- Run controls -------------------------------------------------------------
USE_API        = False         # Force fuzzy-only mode for maximum speed
API_QUERY_MODE = 'strict'      # Placeholder when API is re-enabled
MAX_ROWS       = None          # None = all rows; set e.g. 100 for quick test runs

# -- Quick path check ---------------------------------------------------------
if input_paths:
    print(f'  Batch mode enabled: {len(input_paths)} input files')
    for p in input_paths:
        status = 'OK' if p.exists() else 'NOT FOUND'
        print(f'  {status}  {p}')
else:
    status = 'OK' if Path(INPUT_FILE).exists() else 'NOT FOUND'
    print(f'  {status}  {INPUT_FILE}')

for f in [DIM_LOCATION, ALIAS_FILE]:
    status = 'OK' if Path(f).exists() else 'NOT FOUND'
    print(f'  {status}  {f}')
print(f'  USE_API = {USE_API} (fuzzy-only)')


  OK  ..\..\data\sample\ps_address_mapping_0401.csv
  OK  ..\..\data\mapping\dim_location_20260316_v2.csv
  OK  ..\..\data\utils\ph_address_alias_extended_v4.csv
  USE_API = False (fuzzy-only)


## Cell 2 — Imports & logging setup

Imports required libraries: pandas, numpy, rapidfuzz, logging, and IPython display utilities.

Configures structured logging with timestamps and log levels. Creates a logger instance (`log`) that will be used throughout the pipeline for informational messages.

In [3]:
import re
import time
import logging
import urllib.request
import urllib.parse
import json
from typing import Optional

import pandas as pd
import numpy as np
from IPython.display import display
from rapidfuzz import fuzz, process
from tqdm.notebook import tqdm

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('ph_pipeline')
log.info('Imports OK')


13:14:22  INFO      Imports OK


## Cell 3 — Load reference data

Loads `dim_location` (42 000+ PSGC barangay rows) and `ph_address_alias` (499 alias pairs), then builds fast in-memory lookup lists for fuzzy matching.

In [4]:
def _read_csv_with_fallback(path: str) -> pd.DataFrame:
    strict_encodings = ['utf-8', 'utf-8-sig']
    for enc in strict_encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue

    # Keep mostly-UTF8 files readable even if they contain a few broken bytes.
    try:
        log.warning(f'Loaded {path} using utf-8 with replacement for invalid bytes')
        return pd.read_csv(path, encoding='utf-8', encoding_errors='replace')
    except Exception:
        pass

    for enc in ['cp1252', 'latin1']:
        try:
            log.warning(f'Loaded {path} using fallback encoding: {enc}')
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue

    raise ValueError(f'Unable to decode CSV file: {path}')


def load_reference(dim_path: str, alias_path: str):
    log.info('Loading dim_location ...')
    dim = _read_csv_with_fallback(dim_path)
    str_cols = ['region_name', 'province_name', 'city_name', 'barangay_name']
    for c in str_cols:
        dim[c] = dim[c].fillna('').astype(str).str.upper().str.strip()

    log.info('Loading alias table ...')
    alias = _read_csv_with_fallback(alias_path)
    alias['pattern'] = alias['pattern'].fillna('').astype(str).str.upper().str.strip()
    alias['replacement'] = alias['replacement'].fillna('').astype(str).str.upper().str.strip()

    cities = sorted(x for x in dim['city_name'].unique().tolist() if x)
    provinces = sorted(x for x in dim['province_name'].unique().tolist() if x)
    regions = sorted(x for x in dim['region_name'].unique().tolist() if x)
    barangays = sorted(x for x in dim['barangay_name'].unique().tolist() if x)

    log.info(
        f'dim_location: {len(dim):,} rows | '
        f'{len(cities)} cities | {len(provinces)} provinces | '
        f'{len(regions)} regions | {len(barangays):,} barangays'
    )
    return dim, alias, cities, provinces, regions, barangays


dim, alias_df, cities, provinces, regions, barangays = load_reference(
    DIM_LOCATION, ALIAS_FILE
)

print('\n-- Sample dim_location --')
display(dim.head(3))
print('\n-- Sample aliases --')
display(alias_df.head(10))


13:14:22  INFO      Loading dim_location ...
13:14:22  INFO      Loading alias table ...
13:14:22  WARNING   Loaded ..\..\data\utils\ph_address_alias_extended_v4.csv using utf-8 with replacement for invalid bytes
13:14:22  INFO      dim_location: 42,011 rows | 1426 cities | 83 provinces | 18 regions | 25,843 barangays



-- Sample dim_location --


,psgc_10_digit,region_name,province_name,city_name,barangay_name,region_code,province_code,city_code,barangay_code
0,1400101001,CORDILLERA ADMINISTRATIVE REGION (CAR),ABRA,BANGUED,AGTANGAO,14,1,1,1
1,1400101002,CORDILLERA ADMINISTRATIVE REGION (CAR),ABRA,BANGUED,ANGAD,14,1,1,2
2,1400101003,CORDILLERA ADMINISTRATIVE REGION (CAR),ABRA,BANGUED,BAÑACAO,14,1,1,3



-- Sample aliases --


,pattern,replacement
0,BRGY,BARANGAY
1,BRGY.,BARANGAY
2,BRG,BARANGAY
3,BRG.,BARANGAY
4,BARRIO,BARANGAY
5,BARRIO.,BARANGAY
6,POB,POBLACION
7,POB.,POBLACION
8,POBL,POBLACION
9,POBL.,POBLACION


## Cell 4 — Stage 1: Alias normalization

Builds a **single-pass compiled regex** from all 499 alias pairs (sorted longest-first so multi-word patterns like `CITY OF MARIKINA` fire before the single-token `MARIKINA`).

A post-pass deduplication step catches chained-alias artifacts like `CITY CITY` or `BARANGAY BARANGAY` that arise when both `CITY OF X → X CITY` and `X → X CITY` fire on the same string.

In [5]:
def build_alias_regex(alias_df: pd.DataFrame):
    pairs = [
        (p.strip(), r.strip())
        for p, r in zip(alias_df['pattern'], alias_df['replacement'])
        if isinstance(p, str) and p.strip()
    ]
    pairs.sort(key=lambda x: -len(x[0]))   # longest pattern first
    replace_map = {p: r for p, r in pairs}
    pattern  = '|'.join(re.escape(p) for p, _ in pairs)
    compiled = re.compile(r'\b(' + pattern + r')\b')
    return compiled, replace_map


def normalize_alias(text: str, compiled_re, replace_map: dict) -> str:
    text = str(text).upper().strip()
    # Normalize punctuation first so aliases like 'PQUE.' can match robustly.
    text = re.sub(r'[\.,;:/\\\-\(\)\[\]\{\}]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = compiled_re.sub(lambda m: replace_map.get(m.group(0), m.group(0)), text)
    # Remove duplicate consecutive words (e.g. 'CITY CITY', 'BARANGAY BARANGAY')
    text = re.sub(
        r'\b(CITY|BARANGAY|STREET|AVENUE|BOULEVARD|VILLAGE|PROVINCE)\s+\1\b',
        r'\1', text, flags=re.IGNORECASE
    )
    return re.sub(r'\s+', ' ', text).strip()


compiled_re, replace_map = build_alias_regex(alias_df)
log.info(f'Alias regex built from {len(alias_df)} patterns')

# -- Quick test on sample addresses --------------------------------------------
test_cases = [
    'sapang palay sjdm',
    '112 Ballecer st Zone South Signal Village Taguig',
    '55 Dawn St South Kim View Park SSS Village Concepcionh 2 MArikina City',
    'loilo City',
    'carmencdoc Carmen CAGAYAN DE ORO CITY (Capital) MISAMIS ORIENTAL',
    'LIONS PARK RES SUNVALLEY PQUE.',
]
print(f'{"ORIGINAL":<55}  {"NORMALIZED"}')
print('-' * 110)
for t in test_cases:
    print(f'{t:<55}  {normalize_alias(t, compiled_re, replace_map)}')


13:14:22  INFO      Alias regex built from 512 patterns


ORIGINAL                                                 NORMALIZED
--------------------------------------------------------------------------------------------------------------
sapang palay sjdm                                        SAPANG PALAY SAN JOSE DEL MONTE CITY
112 Ballecer st Zone South Signal Village Taguig         112 BALLECER STREET ZONE SOUTH SIGNAL VILLAGE TAGUIG CITY
55 Dawn St South Kim View Park SSS Village Concepcionh 2 MArikina City  55 DAWN STREET SOUTH KIM VIEW PARK SSS VILLAGE CONCEPCIONH 2 MARIKINA CITY
loilo City                                               LOILO CITY
carmencdoc Carmen CAGAYAN DE ORO CITY (Capital) MISAMIS ORIENTAL  CARMENCDOC CARMEN CAGAYAN DE ORO CITY CAPITAL MISAMIS ORIENTAL
LIONS PARK RES SUNVALLEY PQUE.                           LIONS PARK RES SUNVALLEY PARA�AQUE CITY


## Cell 5 — Stage 2: Fuzzy matching helpers

Two functions used throughout the pipeline:
- **`best_match`** — matches a single query string against a list (full-string scorer)
- **`token_scan`** — splits the address into tokens and tries each token independently; returns the highest-scoring hit across all tokens

Both use `rapidfuzz.fuzz.WRatio` which handles partial overlaps well (e.g. `QUEZON CITY` inside a longer address string).

In [6]:
def best_match(
    query: str,
    choices: list,
    scorer=fuzz.WRatio,
    score_cutoff: int = 0,
) -> tuple[Optional[str], int]:
    """Return (best_match_string, score) or (None, 0) if below cutoff."""
    if not query:
        return None, 0
    result = process.extractOne(query, choices, scorer=scorer,
                                score_cutoff=score_cutoff)
    return (result[0], int(result[1])) if result else (None, 0)


def token_scan(
    tokens: list,
    choices: list,
    score_cutoff: int = 0,
) -> tuple[Optional[str], int]:
    """Try each token against choices; return best (match, score)."""
    best_s, best_m = 0, None
    for tok in tokens:
        if len(tok) < 3:
            continue
        m, s = best_match(tok, choices, score_cutoff=score_cutoff)
        if s > best_s:
            best_s, best_m = s, m
    return best_m, best_s


# ── Quick smoke test ──────────────────────────────────────────────────────────
tests = [
    ('QUEZON CITY', cities),
    ('ILOCOS SUR', provinces),
    ('TAGUIG', cities),
    ('SOUTH SIGNAL', cities),   # should score low / return nothing meaningful
]
print(f'{"Query":<25}  {"Best match":<35}  Score')
print('─' * 75)
for q, lst in tests:
    m, s = best_match(q, lst, score_cutoff=0)
    print(f'{q:<25}  {str(m):<35}  {s}')


Query                      Best match                           Score
───────────────────────────────────────────────────────────────────────────
QUEZON CITY                QUEZON CITY                          100
ILOCOS SUR                 ILOCOS SUR                           100
TAGUIG                     CITY OF TAGUIG                       90
SOUTH SIGNAL               SIGAY                                72


## Cell 6 — Stage 2 + 3: Strict evidence-based matcher & confidence gate

### Strict validity policy
A row is valid only when location evidence is explicit in the address text (after normalization):
- explicit city mention, or
- explicit barangay name that uniquely maps to one city in `dim_location`, or
- explicit curated district/area phrase mapped to a city (exact phrase only).

### Barangay-first recovery (strict)
If city is missing but barangay is explicitly written (e.g., `BRGY SUN VALLEY`) and that barangay uniquely maps to one city in `dim_location`, the row is accepted and city is assigned from `dim_location`.

### No-assumption rule
- No city/area/barangay evidence: invalid (street/building-only addresses remain invalid).
- If barangay is explicitly indicated but cannot be resolved inside the resolved city: invalid.
- If city is explicit and barangay is not mentioned: valid at city level.

### Confidence levels
| Level | Condition | Action |
|---|---|---|
| `high` | explicit city or explicit unique barangay | Accept |
| `medium` | explicit curated area/district cue | Accept with caution |
| `low` | missing/conflicting/unresolved explicit evidence | Invalid |

In [7]:
def match_address(
    address_norm: str,
    dim: pd.DataFrame,
    cities: list, provinces: list, regions: list, barangays: list,
    city_high: int = CITY_SCORE_HIGH,
    city_low: int  = CITY_SCORE_LOW,
    prov_high: int = PROV_SCORE_HIGH,
    prov_low: int  = PROV_SCORE_LOW,
) -> dict:
    result = dict(
        city_name=None, city_code=None, city_score=0,
        province_name=None, province_code=None, province_score=0,
        region_name=None, region_code=None,
        barangay_name=None, barangay_code=None,
        psgc_10_digit=None,
        confidence='none', needs_api=False,
        city_required_pass=False, city_signal=None, invalid_reason=None,
    )

    def _ascii_fold(text: str) -> str:
        # Fold accents so PARAÑAQUE and PARANAQUE match consistently.
        import unicodedata
        txt = unicodedata.normalize('NFKD', str(text))
        return ''.join(ch for ch in txt if not unicodedata.combining(ch))

    def _canon(text: str) -> str:
        text = _ascii_fold(str(text).upper())
        text = re.sub(r'[^A-Z0-9]+', ' ', text)
        return re.sub(r'\s+', ' ', text).strip()

    if not hasattr(match_address, '_cache'):
        dim_work = dim.copy()
        for c in ['region_name', 'province_name', 'city_name', 'barangay_name']:
            dim_work[c] = dim_work[c].fillna('').astype(str).str.upper().str.strip()
        dim_work['_city_canon'] = dim_work['city_name'].apply(_canon)
        dim_work['_prov_canon'] = dim_work['province_name'].apply(_canon)
        dim_work['_brgy_canon'] = dim_work['barangay_name'].apply(_canon)

        city_variant_map = {}
        city_raw_map = {}
        for city_name in sorted(x for x in dim_work['city_name'].unique().tolist() if x):
            base = _canon(city_name)
            city_raw_map[base] = city_name
            variants = {base}
            if base.startswith('CITY OF '):
                core = base[len('CITY OF '):].strip()
                if core:
                    variants.add(core)
                    variants.add(f'{core} CITY')
            if base.startswith('MUNICIPALITY OF '):
                core = base[len('MUNICIPALITY OF '):].strip()
                if core:
                    variants.add(core)
            if base.endswith(' CITY'):
                core = base[:-len(' CITY')].strip()
                if core:
                    variants.add(core)
                    variants.add(f'CITY OF {core}')
            for v in variants:
                if v and v not in city_variant_map:
                    city_variant_map[v] = base

        sorted_city_variants = sorted(city_variant_map.keys(), key=len, reverse=True)

        province_variant_map = {}
        for prov_name in sorted(x for x in dim_work['province_name'].unique().tolist() if x):
            p_can = _canon(prov_name)
            if p_can and p_can not in province_variant_map:
                province_variant_map[p_can] = p_can
        sorted_prov_variants = sorted(province_variant_map.keys(), key=len, reverse=True)

        barangay_values = sorted(
            x for x in dim_work['_brgy_canon'].unique().tolist() if x
        )
        barangay_set = set(barangay_values)
        brgy_token_max = max((len(x.split()) for x in barangay_values), default=1)

        barangay_to_cityset = {}
        for _, r in dim_work[['_brgy_canon', '_city_canon']].drop_duplicates().iterrows():
            b = r['_brgy_canon']
            c = r['_city_canon']
            if b and c:
                barangay_to_cityset.setdefault(b, set()).add(c)

        # Manila district keys resolved from dim city names (no forced CITY OF MANILA).
        manila_district_cores = [
            'TONDO', 'BINONDO', 'SAN NICOLAS', 'SANTA CRUZ', 'QUIAPO',
            'SAMPALOC', 'SAN MIGUEL', 'ERMITA', 'INTRAMUROS', 'MALATE',
            'PACO', 'PANDACAN', 'PORT AREA', 'SANTA ANA',
        ]
        district_city_map = {core: [] for core in manila_district_cores}
        for city_name in sorted(x for x in dim_work['city_name'].unique().tolist() if x):
            c_can = _canon(city_name)
            for core in manila_district_cores:
                core_can = _canon(core)
                if f' {core_can} ' in f' {c_can} ':
                    district_city_map[core].append(city_name)
        district_city_map = {
            k: sorted(set(v))
            for k, v in district_city_map.items()
            if v
        }

        # Curated explicit area cues. Strict exact phrases only.
        area_to_city_raw = {
            # Quezon City
            'CUBAO': ['QUEZON CITY', 'CITY OF QUEZON'],
            'COMMONWEALTH': ['QUEZON CITY', 'CITY OF QUEZON'],
            'DILIMAN': ['QUEZON CITY', 'CITY OF QUEZON'],
            'NEW MANILA': ['QUEZON CITY', 'CITY OF QUEZON'],
            'FAIRVIEW': ['QUEZON CITY', 'CITY OF QUEZON'],
            'NOVALICHES': ['QUEZON CITY', 'CITY OF QUEZON'],
            'PROJECT 4': ['QUEZON CITY', 'CITY OF QUEZON'],
            # Taguig
            'BGC': ['CITY OF TAGUIG', 'TAGUIG'],
            'BONIFACIO GLOBAL CITY': ['CITY OF TAGUIG', 'TAGUIG'],
            'FORT BONIFACIO': ['CITY OF TAGUIG', 'TAGUIG'],
            # Makati
            'LEGASPI VILLAGE': ['CITY OF MAKATI', 'MAKATI CITY', 'MAKATI'],
            'SALCEDO VILLAGE': ['CITY OF MAKATI', 'MAKATI CITY', 'MAKATI'],
            'POBLACION MAKATI': ['CITY OF MAKATI', 'MAKATI CITY', 'MAKATI'],
            # Mandaluyong / San Juan
            'WACK WACK': ['CITY OF MANDALUYONG', 'MANDALUYONG CITY', 'MANDALUYONG'],
            'GREENHILLS': ['CITY OF SAN JUAN', 'SAN JUAN CITY', 'SAN JUAN'],
            # Parañaque / Las Piñas / Pasay / Pasig
            'SUN VALLEY': ['CITY OF PARANAQUE', 'PARANAQUE CITY', 'PARANAQUE'],
            'ALMANZA': ['CITY OF LAS PINAS', 'LAS PINAS CITY', 'LAS PINAS'],
            'BACLARAN': ['PASAY CITY', 'CITY OF PASAY', 'PASAY'],
            'ORTIGAS CENTER': ['PASIG CITY', 'CITY OF PASIG', 'PASIG'],
            # Caloocan / Malabon / Navotas
            'MONUMENTO': ['CALOOCAN CITY', 'CITY OF CALOOCAN', 'CALOOCAN'],
            'POTRERO': ['MALABON CITY', 'CITY OF MALABON', 'MALABON'],
            'TANZA': ['NAVOTAS CITY', 'CITY OF NAVOTAS', 'NAVOTAS'],
        }

        def _resolve_city_candidate(candidates: list[str]) -> str | None:
            for cand in candidates:
                cc = _canon(cand)
                if cc in city_raw_map:
                    return cc
                if cc in city_variant_map:
                    return city_variant_map[cc]
            return None

        area_to_city = {}
        for area, city_cands in area_to_city_raw.items():
            city_resolved = _resolve_city_candidate(city_cands)
            if city_resolved:
                area_to_city[_canon(area)] = city_resolved

        area_terms_sorted = sorted(area_to_city.keys(), key=len, reverse=True)

        ncr_single_word_city_allow = {
            'MANILA', 'QUEZON', 'MAKATI', 'TAGUIG', 'PASIG', 'PASAY',
            'MANDALUYONG', 'CALOOCAN', 'MALABON', 'NAVOTAS', 'MARIKINA',
            'PARANAQUE', 'LAS PINAS', 'SAN JUAN',
        }

        match_address._cache = {
            'dim_work': dim_work,
            'city_variant_map': city_variant_map,
            'city_raw_map': city_raw_map,
            'sorted_city_variants': sorted_city_variants,
            'sorted_prov_variants': sorted_prov_variants,
            'barangay_set': barangay_set,
            'brgy_token_max': brgy_token_max,
            'barangay_to_cityset': barangay_to_cityset,
            'district_city_map': district_city_map,
            'area_to_city': area_to_city,
            'area_terms_sorted': area_terms_sorted,
            'ncr_single_word_city_allow': {_canon(x) for x in ncr_single_word_city_allow},
        }

    cache = match_address._cache
    dim_work = cache['dim_work']
    city_variant_map = cache['city_variant_map']
    city_raw_map = cache['city_raw_map']
    sorted_city_variants = cache['sorted_city_variants']
    sorted_prov_variants = cache['sorted_prov_variants']
    barangay_set = cache['barangay_set']
    brgy_token_max = cache['brgy_token_max']
    barangay_to_cityset = cache['barangay_to_cityset']
    district_city_map = cache['district_city_map']
    area_to_city = cache['area_to_city']
    area_terms_sorted = cache['area_terms_sorted']
    ncr_single_word_city_allow = cache['ncr_single_word_city_allow']

    lookup_text = _canon(address_norm)
    has_city_keyword = bool(re.search(r'\b(CITY|MUNICIPALITY|MUN)\b', lookup_text))
    has_brgy_keyword = bool(re.search(r'\b(BRGY|BARANGAY|BGY)\b', lookup_text))
    has_province_keyword = bool(re.search(r'\b(PROVINCE|PROV)\b', lookup_text))

    def _find_explicit_city(text_canon: str) -> Optional[str]:
        padded = f' {text_canon} '
        for variant in sorted_city_variants:
            if f' {variant} ' not in padded:
                continue
            words = variant.split()
            if 'CITY' in words or 'MUNICIPALITY' in words or len(words) >= 2:
                return city_variant_map[variant]
            # Single-word city evidence is accepted only for controlled city set.
            if variant in ncr_single_word_city_allow or has_city_keyword:
                return city_variant_map[variant]
        return None

    def _find_explicit_province(text_canon: str) -> Optional[str]:
        if not has_province_keyword:
            return None
        padded = f' {text_canon} '
        for p in sorted_prov_variants:
            if f' {p} ' in padded:
                return p
        return None

    def _find_dim_district_city(text_canon: str):
        padded = f' {text_canon} '
        hits = []
        for core, city_candidates in district_city_map.items():
            core_can = _canon(core)
            if f' {core_can} ' not in padded:
                continue
            if len(city_candidates) == 1:
                hits.append(_canon(city_candidates[0]))
            else:
                # If multiple district city labels exist, pick explicit city label mention first.
                city_hits = []
                for c in city_candidates:
                    if f' {_canon(c)} ' in padded:
                        city_hits.append(_canon(c))
                if city_hits:
                    hits.extend(city_hits)
        hits = sorted(set(hits))
        if len(hits) == 1:
            return hits[0], 'dim_mnl_district'
        return None, None

    def _find_explicit_area_city(text_canon: str):
        padded = f' {text_canon} '
        for area in area_terms_sorted:
            if f' {area} ' in padded:
                return area_to_city[area], area
        return None, None

    def _explicit_ngrams(text_canon: str, max_n: int) -> set[str]:
        toks = text_canon.split()
        out = set()
        for n in range(1, max_n + 1):
            for i in range(0, max(0, len(toks) - n + 1)):
                out.add(' '.join(toks[i:i+n]))
        return out

    def _find_explicit_barangays(text_canon: str) -> list[str]:
        grams = _explicit_ngrams(text_canon, brgy_token_max)
        hits = [g for g in grams if g in barangay_set]
        if not has_brgy_keyword:
            # Without BRGY markers, avoid 1-word accidental street token collisions.
            hits = [h for h in hits if len(h.split()) >= 2]
        return sorted(hits, key=len, reverse=True)

    explicit_city_can = _find_explicit_city(lookup_text)
    explicit_prov_can = _find_explicit_province(lookup_text)
    district_city_can, district_source = _find_dim_district_city(lookup_text)
    area_city_can, area_term = _find_explicit_area_city(lookup_text)
    explicit_brgy_hits = _find_explicit_barangays(lookup_text)

    city_signal_can = explicit_city_can
    city_signal_source = 'explicit_city' if explicit_city_can else None
    city_signal_brgy_can = None

    if not city_signal_can and district_city_can:
        city_signal_can = district_city_can
        city_signal_source = district_source

    if not city_signal_can and explicit_brgy_hits:
        for b in explicit_brgy_hits:
            city_set = barangay_to_cityset.get(b, set())
            if len(city_set) == 1:
                city_signal_can = next(iter(city_set))
                city_signal_source = 'explicit_barangay'
                city_signal_brgy_can = b
                break

        if not city_signal_can and area_city_can:
            city_signal_can = area_city_can
            city_signal_source = 'area_exception'

        if not city_signal_can and has_brgy_keyword:
            result.update(
                city_required_pass=False,
                city_signal=None,
                invalid_reason='barangay_ambiguous_no_city',
                confidence='low',
                needs_api=False,
            )
            return result

    if not city_signal_can and area_city_can:
        city_signal_can = area_city_can
        city_signal_source = 'area_exception'

    if not city_signal_can:
        result.update(
            city_required_pass=False,
            city_signal=None,
            invalid_reason='missing_explicit_city',
            confidence='low',
            needs_api=False,
        )
        return result

    city_subset = dim_work[dim_work['_city_canon'] == city_signal_can].copy()
    if city_subset.empty:
        result.update(
            city_required_pass=False,
            city_signal=city_raw_map.get(city_signal_can, city_signal_can),
            invalid_reason='city_not_resolved',
            confidence='low',
            needs_api=False,
        )
        return result

    if explicit_prov_can:
        city_provs = set(city_subset['_prov_canon'].dropna().unique().tolist())
        if explicit_prov_can not in city_provs:
            result.update(
                city_required_pass=False,
                city_signal=city_raw_map.get(city_signal_can, city_signal_can),
                invalid_reason='province_city_conflict',
                confidence='low',
                needs_api=False,
            )
            return result
        city_subset = city_subset[city_subset['_prov_canon'] == explicit_prov_can].copy()
        if city_subset.empty:
            result.update(
                city_required_pass=False,
                city_signal=city_raw_map.get(city_signal_can, city_signal_can),
                invalid_reason='province_city_conflict',
                confidence='low',
                needs_api=False,
            )
            return result

    has_brgy_hint = has_brgy_keyword or bool(explicit_brgy_hits)
    city_brgy_set = set(x for x in city_subset['_brgy_canon'].unique().tolist() if x)

    explicit_brgy_in_city = None
    for b in explicit_brgy_hits:
        if b in city_brgy_set:
            explicit_brgy_in_city = b
            break

    if city_signal_source == 'explicit_barangay' and city_signal_brgy_can and not explicit_brgy_in_city:
        explicit_brgy_in_city = city_signal_brgy_can

    if has_brgy_hint and not explicit_brgy_in_city:
        chosen = city_subset.iloc[0]
        result.update(
            city_name=chosen['city_name'],
            city_code=int(chosen['city_code']),
            city_score=100 if city_signal_source in {'explicit_city', 'explicit_barangay', 'dim_mnl_district'} else 95,
            province_name=chosen['province_name'],
            province_code=int(chosen['province_code']),
            province_score=100 if explicit_prov_can else 90,
            region_name=chosen['region_name'],
            region_code=int(chosen['region_code']),
            city_required_pass=False,
            city_signal=chosen['city_name'],
            invalid_reason='barangay_unresolved',
            confidence='low',
            needs_api=False,
        )
        return result

    chosen = city_subset.iloc[0]
    if explicit_brgy_in_city:
        brgy_rows = city_subset[city_subset['_brgy_canon'] == explicit_brgy_in_city]
        if not brgy_rows.empty:
            chosen = brgy_rows.iloc[0]

    result.update(
        city_name=chosen['city_name'],
        city_code=int(chosen['city_code']),
        city_score=100 if city_signal_source in {'explicit_city', 'explicit_barangay', 'dim_mnl_district'} else 95,
        province_name=chosen['province_name'],
        province_code=int(chosen['province_code']),
        province_score=100 if explicit_prov_can else 90,
        region_name=chosen['region_name'],
        region_code=int(chosen['region_code']),
        barangay_name=chosen['barangay_name'] if explicit_brgy_in_city else None,
        barangay_code=int(chosen['barangay_code']) if explicit_brgy_in_city else None,
        psgc_10_digit=int(chosen['psgc_10_digit']),
        confidence='high' if city_signal_source in {'explicit_city', 'explicit_barangay', 'dim_mnl_district'} else 'medium',
        needs_api=False,
        city_required_pass=True,
        city_signal=chosen['city_name'],
        invalid_reason=None,
    )
    return result


# -- Generalized strict-evidence regression tests -------------------------------
test_addresses = [
    '19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY',
    '112 Ballecer st Zone South Signal Village Taguig',
    'LIONS PARK RES SUN VALLEY',
    'House 4 Adolfo Compound, Highway 11 St Brgy Talamban, City of Cebu',
    'BGC 30TH STREET',
    'TONDO MANILA',
    'BINONDO MANILA',
    '2248 Angel Linao St',
    'AURORA BLVD COR 20TH AVE',
    'NEW MANILA near E. Rodriguez',
    'BRGY SUN VALLEY',
    'BRGY STA CRUZ',
    '123 5TH FLOOR ABC BUILDING',
    'CUBAO',
]

for idx, raw in enumerate(test_addresses, 1):
    addr = normalize_alias(raw, compiled_re, replace_map)
    ans = match_address(addr, dim, cities, provinces, regions, barangays)
    print(f'Test {idx}')
    print(f'  Input      : {raw}')
    print(f'  Normalized : {addr}')
    print(f'  City signal: {ans["city_signal"]}')
    print(f'  City       : {ans["city_name"]}')
    print(f'  Province   : {ans["province_name"]}')
    print(f'  Barangay   : {ans["barangay_name"]}')
    print(f'  Confidence : {ans["confidence"]}')
    print(f'  Invalid rsn: {ans["invalid_reason"]}')
    print('-' * 80)

Test 1
  Input      : 19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY
  Normalized : 19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY
  City signal: QUEZON CITY
  City       : QUEZON CITY
  Province   : METRO MANILA
  Barangay   : BATASAN HILLS
  Confidence : high
  Invalid rsn: None
--------------------------------------------------------------------------------
Test 2
  Input      : 112 Ballecer st Zone South Signal Village Taguig
  Normalized : 112 BALLECER STREET ZONE SOUTH SIGNAL VILLAGE TAGUIG CITY
  City signal: CITY OF TAGUIG
  City       : CITY OF TAGUIG
  Province   : METRO MANILA
  Barangay   : SOUTH SIGNAL VILLAGE
  Confidence : high
  Invalid rsn: None
--------------------------------------------------------------------------------
Test 3
  Input      : LIONS PARK RES SUN VALLEY
  Normalized : LIONS PARK RES SUN VALLEY
  City signal: CITY OF PARAÑAQUE
  City       : CITY OF PARAÑAQUE
  Province   : METRO MANILA
  Barangay   : SUN VALLEY
  Confidence 

## Cell 7 — Run Stage 1–3 on all input rows

Loads the input file, applies alias normalization, then runs the fuzzy matcher on every row. Results are split into `high_conf` (no API needed) and `low_conf` (needs Nominatim verification).

In [8]:
import time

t_start = time.time()

def _read_input_table(path):
    p = Path(path)
    if p.suffix.lower() == '.csv':
        return pd.read_csv(p)
    return pd.read_excel(p)

if input_paths:
    existing_paths = [p for p in input_paths if p.exists()]
    missing_paths = [p for p in input_paths if not p.exists()]

    for mp in missing_paths:
        log.warning(f'Missing batch file: {mp}')

    if not existing_paths:
        raise FileNotFoundError('No batch files found in input_paths.')

    log.info(f'Loading batch files: {len(existing_paths)} found')
    frames = []
    for p in existing_paths:
        tmp = _read_input_table(p)
        tmp['_batch_file'] = p.name
        frames.append(tmp)
    df_input = pd.concat(frames, ignore_index=True)
else:
    log.info(f'Loading {INPUT_FILE} ...')
    df_input = _read_input_table(INPUT_FILE)

if MAX_ROWS:
    df_input = df_input.iloc[:MAX_ROWS]
    log.info(f'Limiting to {MAX_ROWS} rows (MAX_ROWS is set)')

addr_col = df_input.columns[0]
log.info(f'Address column: "{addr_col}"  |  Total rows: {len(df_input):,}')

# -- Stage 1: alias normalization ---------------------------------------------
log.info('Stage 1: alias normalization ...')
addresses = df_input[addr_col].fillna('').astype(str).tolist()
normalized = [normalize_alias(x, compiled_re, replace_map) for x in addresses]
df_input['address_normalized'] = normalized

# -- Stage 2-3: fuzzy match + confidence gate ---------------------------------
log.info('Stage 2-3: fuzzy matching + confidence gate ...')
records = []
batch_files = df_input['_batch_file'].tolist() if '_batch_file' in df_input.columns else None

for i, addr_norm in enumerate(tqdm(normalized, total=len(normalized), desc='Fuzzy match', unit='row')):
    rec = {
        'original_address': addresses[i],
        'address_normalized': addr_norm,
        'api_status': 'skipped',
        'osm_city': None, 'osm_province': None, 'osm_region': None,
        'osm_display': None, 'osm_lat': None, 'osm_lon': None,
    }
    if batch_files is not None:
        rec['batch_file'] = batch_files[i]
    rec.update(match_address(
        addr_norm,
        dim, cities, provinces, regions, barangays,
    ))
    rec['needs_api'] = False
    records.append(rec)

high_conf = [r for r in records if r.get('confidence') == 'high']
low_conf = [r for r in records if r.get('confidence') != 'high']

t_fuzzy = time.time() - t_start
log.info(f'Fuzzy pass done in {t_fuzzy:.1f}s')
log.info(f'  High confidence : {len(high_conf):,}')
log.info(f'  Lower confidence: {len(low_conf):,}')
log.info('  API stage is disabled; fuzzy-only output will be used')

# -- Preview ------------------------------------------------------------------
preview_cols = [
    'original_address', 'address_normalized',
    'city_name', 'province_name', 'confidence', 'city_score'
]
if records and 'batch_file' in records[0]:
    preview_cols = ['batch_file'] + preview_cols

df_preview = pd.DataFrame(records)[preview_cols]
display(df_preview)


13:14:26  INFO      Loading ..\..\data\sample\ps_address_mapping_0401.csv ...
13:14:26  INFO      Address column: "original_address"  |  Total rows: 224,121
13:14:26  INFO      Stage 1: alias normalization ...
13:14:33  INFO      Stage 2-3: fuzzy matching + confidence gate ...


Fuzzy match:   0%|          | 0/224121 [00:00<?, ?row/s]

13:19:56  INFO      Fuzzy pass done in 329.6s
13:19:56  INFO        High confidence : 143,578
13:19:56  INFO        Lower confidence: 80,543
13:19:56  INFO        API stage is disabled; fuzzy-only output will be used


,original_address,address_normalized,city_name,province_name,confidence,city_score
0,Unit 5 erica Rhianne Bldg. one Santana Place B...,UNIT 5 ERICA RHIANNE BUILDING ONE SANTANA PLAC...,ABUCAY,BATAAN,high,100
1,L10 B23 SAN BENIGNO ST METROCOR SUBD,L10 B23 SAN BENIGNO STREET METROCOR SUBDIVISION,AGLIPAY,QUIRINO,high,100
2,002 PUROK UNO BRGY PRINZA,002 PUROK UNO BARANGAY PRINZA,AGOO,LA UNION,high,100
3,purok 8 barangay 4,PUROK 8 BARANGAY 4,AGOO,LA UNION,high,100
4,KALYE MASAYA PUROK KAAKBAYAN BRGY TINIGUIBAN G...,KALYE MASAYA PUROK KAAKBAYAN BARANGAY TINIGUIB...,AGOO,LA UNION,high,100
...,...,...,...,...,...,...
224116,"groundfloor,ManulifePhilippines,RositaBuilding...",GROUNDFLOOR MANULIFEPHILIPPINES ROSITABUILDING...,CITY OF CABANATUAN,NUEVA ECIJA,high,100
224117,ManulifeOfficebrgy Zulueta District (Pob.) CAB...,MANULIFEOFFICEBRGY ZULUETA DISTRICT POBLACION ...,CITY OF CABANATUAN,NUEVA ECIJA,high,100
224118,N/A Zulueta District (Pob.) CABANATUAN CITY NU...,N A ZULUETA DISTRICT POBLACION CABANATUAN CITY...,CITY OF CABANATUAN,NUEVA ECIJA,high,100
224119,37VALERIANOST Zulueta District (Pob.) CABANATU...,37VALERIANOST ZULUETA DISTRICT POBLACION CABAN...,CITY OF CABANATUAN,NUEVA ECIJA,high,100


In [9]:
# Snapshot fuzzy outputs before API mutates records
pre_api_df = pd.DataFrame([dict(r) for r in records])
print(f'Captured pre_api_df rows: {len(pre_api_df):,}')

Captured pre_api_df rows: 224,121


## Cell 8 -- API stage disabled

API verification is intentionally disabled in this fast run profile.

This notebook now runs in fuzzy-only mode using:
- `dim_location` for PSGC hierarchy mapping
- `ph_address_alias_extended_v4.csv` for normalization

If API is needed later, re-enable it in configuration and restore the API stage logic.


In [10]:
# API stage intentionally disabled for maximum throughput in fuzzy-only mode.
# Keep this cell as a no-op so downstream cells continue to run without changes.

USE_API = False
log.info('API stage skipped (fuzzy-only mode).')

for row in low_conf:
    row['api_status'] = 'skipped'
    row['needs_api'] = False

log.info('Proceeding directly to merge/output stages.')


13:19:58  INFO      API stage skipped (fuzzy-only mode).


13:19:58  INFO      Proceeding directly to merge/output stages.


## Cell 9 — Stage 5a: Merge results & determine validity

Merges the fuzzy-match results from Stage 2-3 and applies the final validity rule.

**Validity Rules:**
- Address is **valid** when city evidence is present in the address (`city_required_pass=True`)
- Rows without explicit city evidence are marked **invalid** (`missing_explicit_city`)
- If city evidence exists but `dim_location` mapping is incomplete, row is still valid with `invalid_reason='city_not_resolved'` for diagnostics

**Strict no-assumption policy:** Street/building addresses with no geographic reference are rejected.

Outputs summary statistics: confidence distribution, API status breakdown, and invalid reason counts.

In [11]:
all_records = low_conf if USE_API else (high_conf + low_conf)
out_df = pd.DataFrame(all_records)

if 'city_required_pass' not in out_df.columns:
    out_df['city_required_pass'] = False
out_df['city_required_pass'] = out_df['city_required_pass'].fillna(False).astype(bool)

# Final validity flag with strict city-evidence requirement
if USE_API:
    out_df['is_valid'] = out_df['city_required_pass']
else:
    out_df['is_valid'] = out_df['city_required_pass']

# Confidence value counts
print('Confidence distribution:')
print(out_df['confidence'].value_counts().to_string())
if 'api_status' in out_df.columns:
    print('\nAPI status distribution:')
    print(out_df['api_status'].value_counts().to_string())
if 'invalid_reason' in out_df.columns:
    print('\nInvalid reason distribution:')
    print(out_df['invalid_reason'].fillna('resolved').value_counts().to_string())
print(f'\nValid   : {out_df["is_valid"].sum():,}')
print(f'Invalid : {(~out_df["is_valid"]).sum():,}')
print(f'Total   : {len(out_df):,}')


Confidence distribution:
confidence
high      143578
low        78388
medium      2155

API status distribution:
api_status
skipped    224121

Invalid reason distribution:
invalid_reason
resolved                      145733
barangay_unresolved            47693
missing_explicit_city          29321
barangay_ambiguous_no_city      1361
province_city_conflict            13

Valid   : 145,733
Invalid : 78,388
Total   : 224,121


In [12]:
# City-only sanity check: barangay should remain blank when not explicit
city_only_mask = out_df['original_address'].astype(str).str.lower().isin(['vigan ilocos sur', 'loilo city'])
display(out_df.loc[city_only_mask, [
    'original_address', 'city_name', 'province_name', 'barangay_name', 'barangay_code'
]])

,original_address,city_name,province_name,barangay_name,barangay_code
28347,VIGAN ILOCOS SUR,CITY OF VIGAN,ILOCOS SUR,NaN,NaN
28349,vigan ilocos sur,CITY OF VIGAN,ILOCOS SUR,NaN,NaN
209071,loilo City,NaN,NaN,NaN,NaN


In [13]:
# Detailed-address check: barangay should still be populated when appropriate
detail_mask = out_df['original_address'].astype(str).str.contains(
    'Batasan|South Signal|Kim View|sapang palay', case=False, na=False
)
display(out_df.loc[detail_mask, [
    'original_address', 'city_name', 'province_name', 'barangay_name', 'barangay_code'
]])

,original_address,city_name,province_name,barangay_name,barangay_code
105,Brgy. Sapang Palay,BATAN,AKLAN,PALAY,17.0
107,Lot 26 blk 406 Phase 1 Heritage Villas Metroga...,BATAN,AKLAN,PALAY,17.0
1591,Brgy sapang palay sjdm,CITY OF SAN JOSE DEL MONTE,BULACAN,SAPANG PALAY,10.0
1609,Brgy Sapang palay SJDM,CITY OF SAN JOSE DEL MONTE,BULACAN,SAPANG PALAY,10.0
1674,brgy citrus sapang palay sjdm,CITY OF SAN JOSE DEL MONTE,BULACAN,SAPANG PALAY,10.0
...,...,...,...,...,...
189125,"Marikina Carwash Hub - QC BRANCH, Batasan-San ...",CITY OF MARIKINA,METRO MANILA,NaN,NaN
202447,FELICIDAD PHASE2 BATASAN SAN MATEO RIZAL,SAN MATEO,ISABELA,NaN,NaN
202911,Batasan HIlls San Mateo Rizal,SAN MATEO,ISABELA,NaN,NaN
204966,Blk 69 Lot 4-7 San Rafael IV Area H phase 2 sa...,SAN RAFAEL,BULACAN,NaN,NaN


In [14]:
# Before-vs-after change report (fuzzy baseline vs API-final)
compare_cols = ['city_name', 'province_name', 'confidence', 'api_status']
base = pre_api_df[['original_address'] + compare_cols].copy()
base = base.rename(columns={
    'city_name': 'city_before',
    'province_name': 'province_before',
    'confidence': 'confidence_before',
    'api_status': 'api_status_before',
})

final = out_df[['original_address'] + compare_cols].copy()
final = final.rename(columns={
    'city_name': 'city_after',
    'province_name': 'province_after',
    'confidence': 'confidence_after',
    'api_status': 'api_status_after',
})

change_df = base.merge(final, on='original_address', how='inner')
changed_mask = (
    change_df['city_before'].fillna('') != change_df['city_after'].fillna('')
) | (
    change_df['province_before'].fillna('') != change_df['province_after'].fillna('')
) | (
    change_df['confidence_before'].fillna('') != change_df['confidence_after'].fillna('')
) | (
    change_df['api_status_before'].fillna('') != change_df['api_status_after'].fillna('')
)

changed_rows = change_df[changed_mask].reset_index(drop=True)
print(f'Rows changed after API verification: {len(changed_rows):,} / {len(change_df):,}')
display(changed_rows)

Rows changed after API verification: 0 / 224,821


,original_address,city_before,province_before,confidence_before,api_status_before,city_after,province_after,confidence_after,api_status_after


In [15]:
# Compact delta view
compact = changed_rows[[
    'original_address',
    'city_before', 'city_after',
    'province_before', 'province_after',
    'api_status_before', 'api_status_after'
]].copy()
compact['original_address'] = compact['original_address'].astype(str).str.slice(0, 70)
display(compact)

# Highlight the user's example row
mask = changed_rows['original_address'].astype(str).str.contains('General Espino South Signal', case=False, na=False)
if mask.any():
    print('General Espino South Signal change:')
    display(changed_rows.loc[mask, [
        'original_address',
        'city_before', 'city_after',
        'province_before', 'province_after',
        'confidence_before', 'confidence_after',
        'api_status_before', 'api_status_after'
    ]])

,original_address,city_before,city_after,province_before,province_after,api_status_before,api_status_after


## Cell 10 — Stage 5b: Export to Excel

Exports the pipeline results to three Excel files with color-coded sheets:

**Output Files:**
1. `verified_addresses_parts_XXX_YYY.xlsx` (green tab) — Valid addresses only with base hierarchy columns
2. `invalid_addresses_parts_XXX_YYY.xlsx` (red tab) — Invalid addresses with same base columns
3. `combined_addresses_parts_XXX_YYY.xlsx` (blue tab) — All addresses with full diagnostics

**Batch Naming:** File suffix (e.g., `parts_001_010`) is auto-detected from input batch file names.

**Column Sets:**
- Base columns: Original Address, Barangay Code/Name, City Code/Name, Province Code/Name, Region Code/Region Name
- Combined file adds: Normalized Address, PSGC code, Confidence level, City/Province scores, OSM location data, Status

Rows are automatically widthened to fit content (max 42 chars) with frozen header rows.

In [16]:
from pathlib import Path

# -- Output targets ------------------------------------------------------------
out_root = Path(BASE_DIR) / 'output'
out_verified_dir = out_root / 'verified'
out_invalid_dir = out_root / 'invalid'
out_verified_dir.mkdir(parents=True, exist_ok=True)
out_invalid_dir.mkdir(parents=True, exist_ok=True)

# Build suffix from detected input parts (e.g. parts_001_010)
batch_nums = []
for p in input_paths:
    m = re.search(r'part_(\d+)', p.name.lower())
    if m:
        batch_nums.append(int(m.group(1)))

if batch_nums:
    batch_suffix = f'parts_{min(batch_nums):03d}_{max(batch_nums):03d}'
elif input_paths:
    batch_suffix = f'batch_{len(input_paths):03d}'
else:
    batch_suffix = 'single'

VERIFIED_FILE = str(out_verified_dir / f'verified_addresses_{batch_suffix}.xlsx')
INVALID_FILE = str(out_invalid_dir / f'invalid_addresses_{batch_suffix}.xlsx')
COMBINED_FILE = str(out_root / f'combined_addresses_{batch_suffix}.xlsx')

# -- Column order required by business template (image) -----------------------
BASE_EXPORT_COLS = [
    'Original_Address',
    'barangay_code', 'barangay',
    'city_code', 'city',
    'province_code', 'province',
    'region_code', 'region_name',
]

# Build standardized base table from pipeline output
base_df = out_df.copy()
base_df = base_df.rename(columns={
    'original_address': 'Original_Address',
    'barangay_name': 'barangay',
    'city_name': 'city',
    'province_name': 'province',
})

# Ensure expected columns exist even when null in invalid rows
for c in BASE_EXPORT_COLS:
    if c not in base_df.columns:
        base_df[c] = None

verified_df = base_df[base_df['is_valid']][BASE_EXPORT_COLS].reset_index(drop=True)
invalid_df = base_df[~base_df['is_valid']][BASE_EXPORT_COLS].reset_index(drop=True)

# Combined file keeps additional diagnostics + status
combined_cols = [
    'Original_Address',
    'Normalized_Address',
    'barangay_code', 'barangay',
    'city_code', 'city',
    'province_code', 'province',
    'region_code', 'region_name',
    'psgc_10',
    'confidence',
    'city_score',
    'province_score',
    'osm_display',
    'osm_lat',
    'osm_lon',
    'status',
]

combined_df = base_df.copy()
combined_df['Normalized_Address'] = combined_df.get('address_normalized')
combined_df['psgc_10'] = combined_df.get('psgc_10_digit')
combined_df['status'] = np.where(combined_df['is_valid'], 'verified', 'invalid')
for c in combined_cols:
    if c not in combined_df.columns:
        combined_df[c] = None
combined_df = combined_df[combined_cols].reset_index(drop=True)


def write_xlsx(df: pd.DataFrame, path: str, sheet_name: str,
               tab_color: str, font_color: str = '#000000'):
    with pd.ExcelWriter(path, engine='xlsxwriter') as writer:
        df.to_excel(writer, index=False, sheet_name=sheet_name)
        wb = writer.book
        ws = writer.sheets[sheet_name]
        ws.set_tab_color(tab_color)
        hdr_fmt = wb.add_format({
            'bold': True, 'bg_color': tab_color,
            'font_color': font_color,
            'border': 1, 'text_wrap': True,
            'font_name': 'Arial', 'font_size': 10,
        })
        for i, col in enumerate(df.columns):
            ws.write(0, i, col, hdr_fmt)
            if df.empty:
                max_w = len(col) + 4
            else:
                col_max = df[col].dropna().astype(str).str.len().max()
                if pd.isna(col_max):
                    col_max = len(col)
                max_w = max(int(col_max), len(col)) + 4
            ws.set_column(i, i, min(int(max_w), 42))
        ws.freeze_panes(1, 0)
    log.info(f'Wrote {len(df):,} rows -> {path}')


write_xlsx(verified_df, VERIFIED_FILE, 'Verified', '#00B050')
write_xlsx(invalid_df, INVALID_FILE, 'Invalid', '#FF0000', font_color='#FFFFFF')
write_xlsx(combined_df, COMBINED_FILE, 'Combined', '#1F4E78', font_color='#FFFFFF')

elapsed = time.time() - t_start
print(f'\nTotal elapsed: {elapsed:.1f}s  ({elapsed/60:.2f} min)')
print('Output files:')
print(f'  ✅  {VERIFIED_FILE}  ({len(verified_df):,} rows)')
print(f'  ⚠️   {INVALID_FILE}   ({len(invalid_df):,} rows)')
print(f'  📄  {COMBINED_FILE}  ({len(combined_df):,} rows)')


13:20:14  INFO      Wrote 145,733 rows -> ..\..\data\output\verified\verified_addresses_single.xlsx
13:20:21  INFO      Wrote 78,388 rows -> ..\..\data\output\invalid\invalid_addresses_single.xlsx
13:21:02  INFO      Wrote 224,121 rows -> ..\..\data\output\combined_addresses_single.xlsx



Total elapsed: 396.4s  (6.61 min)
Output files:
  ✅  ..\..\data\output\verified\verified_addresses_single.xlsx  (145,733 rows)
  ⚠️   ..\..\data\output\invalid\invalid_addresses_single.xlsx   (78,388 rows)
  📄  ..\..\data\output\combined_addresses_single.xlsx  (224,121 rows)


## Cell 11 — Results summary

Displays a comprehensive pipeline summary:

**Summary Statistics:**
- Total input rows processed
- Count and percentage of verified vs. invalid addresses
- Confidence distribution breakdown (high, medium, low, none)
- Average city fuzzy-match score for high-confidence rows
- Total elapsed time

**Data Preview:**
- First 10 rows of verified addresses
- All invalid addresses (for inspection)

This provides a quick sanity check to confirm the pipeline executed correctly and shows the address resolution success rate.

In [17]:
print('═' * 80)
print('  PIPELINE SUMMARY')
print('═' * 80)
total = len(out_df)
n_v   = len(verified_df)
n_i   = len(invalid_df)
print(f'  Input rows        : {total:>8,}')
print(f'  Verified          : {n_v:>8,}  ({100*n_v/total:.1f}%)')
print(f'  Invalid           : {n_i:>8,}  ({100*n_i/total:.1f}%)')
print()
print('  Confidence breakdown:')
for conf, cnt in out_df['confidence'].value_counts().items():
    print(f'    {conf:<20} {cnt:>6,}  ({100*cnt/total:.1f}%)')
print()
high_c = out_df[out_df['confidence'] == 'high']
if len(high_c):
    print(f'  Avg city score (high-conf): {high_c["city_score"].mean():.1f}')
print(f'  Total elapsed     : {elapsed:.1f}s')
print('═' * 80)
print()
print('── Verified (first 10 rows) ──')
display(verified_df.head(10))
print()
print('── Invalid (all rows) ──')
display(invalid_df)


════════════════════════════════════════════════════════════════════════════════
  PIPELINE SUMMARY
════════════════════════════════════════════════════════════════════════════════
  Input rows        :  224,121
  Verified          :  145,733  (65.0%)
  Invalid           :   78,388  (35.0%)

  Confidence breakdown:
    high                 143,578  (64.1%)
    low                  78,388  (35.0%)
    medium                2,155  (1.0%)

  Avg city score (high-conf): 100.0
  Total elapsed     : 396.4s
════════════════════════════════════════════════════════════════════════════════

── Verified (first 10 rows) ──


,Original_Address,barangay_code,barangay,city_code,city,province_code,province,region_code,region_name
0,Unit 5 erica Rhianne Bldg. one Santana Place B...,2.0,CALAYLAYAN,1.0,ABUCAY,8.0,BATAAN,3.0,REGION III (CENTRAL LUZON)
1,L10 B23 SAN BENIGNO ST METROCOR SUBD,23.0,SAN BENIGNO,1.0,AGLIPAY,57.0,QUIRINO,2.0,REGION II (CAGAYAN VALLEY)
2,002 PUROK UNO BRGY PRINZA,9.0,PUROK,1.0,AGOO,33.0,LA UNION,1.0,REGION I (ILOCOS REGION)
3,purok 8 barangay 4,9.0,PUROK,1.0,AGOO,33.0,LA UNION,1.0,REGION I (ILOCOS REGION)
4,KALYE MASAYA PUROK KAAKBAYAN BRGY TINIGUIBAN G...,9.0,PUROK,1.0,AGOO,33.0,LA UNION,1.0,REGION I (ILOCOS REGION)
5,"Blk 3, Lot 10 Purok Estrella Brgy Calumpang",9.0,PUROK,1.0,AGOO,33.0,LA UNION,1.0,REGION I (ILOCOS REGION)
6,NHA Ave Purok Silangan Brgy Dela Paz,9.0,PUROK,1.0,AGOO,33.0,LA UNION,1.0,REGION I (ILOCOS REGION)
7,"AA COBRA FARM, PUROK 7 BRGY TANGOB",9.0,PUROK,1.0,AGOO,33.0,LA UNION,1.0,REGION I (ILOCOS REGION)
8,Purok 7 Barrio Site 527,9.0,PUROK,1.0,AGOO,33.0,LA UNION,1.0,REGION I (ILOCOS REGION)
9,Purok Pagsisikap Brgy. Gulang-gulang,9.0,PUROK,1.0,AGOO,33.0,LA UNION,1.0,REGION I (ILOCOS REGION)



── Invalid (all rows) ──


,Original_Address,barangay_code,barangay,city_code,city,province_code,province,region_code,region_name
0,48B AQURIUS ST PAMPLONA DOS LASPINAS CITY,NaN,NaN,18.0,PAMPLONA,15.0,CAGAYAN,2.0,REGION II (CAGAYAN VALLEY)
1,3314 Apitong San Martin De Porres Pque,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,475 malugay st san Martin de porres pque city,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5225 San Lazaro St. San Antonio Valley5 Pque 1715,NaN,NaN,17.0,SAN ANTONIO,48.0,NORTHERN SAMAR,8.0,REGION VIII (EASTERN VISAYAS)
4,370 malugay st. San martin de Porres pque,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
78383,CABATUAN ILOILO Zone VI Pob. (Barangay 6 ) CAB...,NaN,NaN,22.0,CITY OF ILOILO,30.0,ILOILO,6.0,REGION VI (WESTERN VISAYAS)
78384,"Cabatuan, Iloilo Zone VI Pob. (Barangay 6 ) CA...",NaN,NaN,22.0,CITY OF ILOILO,30.0,ILOILO,6.0,REGION VI (WESTERN VISAYAS)
78385,iloilo Zone VII Pob. (Barangay 7) CABATUAN IL...,NaN,NaN,22.0,CITY OF ILOILO,30.0,ILOILO,6.0,REGION VI (WESTERN VISAYAS)
78386,Zone 8 Bulan Sorsogon,NaN,NaN,16.0,CITY OF SORSOGON,62.0,SORSOGON,5.0,REGION V (BICOL REGION)


## Cell 12 — One-shot re-run helper

Defines a reusable `run_pipeline()` function for quick re-runs after config changes.

**Function Parameters:**
- `input_file` (default: INPUT_FILE) — Address file to process
- `dim_path` (default: DIM_LOCATION) — PSGC barangay hierarchy file
- `alias_path` (default: ALIAS_FILE) — Address alias normalization file
- `out_verified`, `out_invalid` — Output file paths
- `max_rows` — Process only N rows (None = all)
- `use_api` — API verification toggle (always False in this profile)
- `api_query_mode` — Search mode if API re-enabled

**Workflow:** Re-uses all functions defined in Cells 3–6 without reloading reference data.

**Usage:** Uncomment the `run_pipeline()` calls at the bottom and execute the cell to re-run the full pipeline.

Do not re-run Cells 3–6 unless reference files (dim_location or aliases) change.

In [18]:
def run_pipeline(
    input_file   = INPUT_FILE,
    dim_path     = DIM_LOCATION,
    alias_path   = ALIAS_FILE,
    out_verified = OUT_VERIFIED,
    out_invalid  = OUT_INVALID,
    max_rows     = MAX_ROWS,
    use_api      = False,
    api_query_mode = API_QUERY_MODE,
):
    t0 = time.time()
    _dim, _alias, _cities, _provinces, _regions, _barangays = load_reference(
        dim_path, alias_path
    )
    _re, _rmap = build_alias_regex(_alias)

    in_path = Path(input_file)
    if in_path.suffix.lower() == '.csv':
        df = pd.read_csv(in_path)
    else:
        df = pd.read_excel(in_path)

    if max_rows:
        df = df.iloc[:max_rows]
    col = df.columns[0]

    addresses = df[col].fillna('').astype(str).tolist()
    normalized = [normalize_alias(x, _re, _rmap) for x in addresses]

    recs = []
    for i, addr_norm in enumerate(tqdm(normalized, total=len(normalized), desc='Fuzzy', unit='row')):
        rec = {
            'original_address': addresses[i],
            'address_normalized': addr_norm,
            'api_status': 'skipped',
            'osm_city': None, 'osm_province': None, 'osm_region': None,
            'osm_display': None, 'osm_lat': None, 'osm_lon': None,
        }
        rec.update(match_address(
            addr_norm,
            _dim, _cities, _provinces, _regions, _barangays,
        ))
        rec['needs_api'] = False
        recs.append(rec)

    if use_api:
        log.warning('API path is disabled in this fast profile; running fuzzy-only instead')

    merged = pd.DataFrame(recs)
    if 'city_required_pass' not in merged.columns:
        merged['city_required_pass'] = False
    merged['city_required_pass'] = merged['city_required_pass'].fillna(False).astype(bool)

    # Strict rule: valid only when city evidence is present in address.
    merged['is_valid'] = merged['city_required_pass']

    base_cols = [
        'original_address',
        'barangay_code', 'barangay_name',
        'city_code', 'city_name',
        'province_code', 'province_name',
        'region_code', 'region_name',
    ]
    base_cols = [c for c in base_cols if c in merged.columns]

    v_df = merged[merged['is_valid']][base_cols].reset_index(drop=True)
    i_df = merged[~merged['is_valid']][base_cols].reset_index(drop=True)

    write_xlsx(v_df, out_verified, 'Verified', '#00B050')
    write_xlsx(i_df, out_invalid, 'Invalid', '#FF0000', font_color='#FFFFFF')

    elapsed = time.time() - t0
    log.info(f'Done in {elapsed:.1f}s  |  Verified: {len(v_df):,}  Invalid: {len(i_df):,}')
    return v_df, i_df


# Uncomment to run:
# verified_df, invalid_df = run_pipeline()
# verified_df, invalid_df = run_pipeline(max_rows=1000)


In [19]:
# Cell 13 -- Invalid pattern diagnostics (batch profile)
diag_df = out_df.copy()
diag_df['orig'] = diag_df['original_address'].fillna('').astype(str)
diag_df['norm'] = diag_df['address_normalized'].fillna('').astype(str)
diag_df['invalid_reason'] = diag_df.get('invalid_reason', pd.Series([None] * len(diag_df))).fillna('resolved')

invalid_only = diag_df[~diag_df['is_valid']].copy()
print(f'Total rows: {len(diag_df):,}')
print(f'Invalid rows: {len(invalid_only):,}')

print('\nTop invalid reasons:')
print(invalid_only['invalid_reason'].value_counts().head(10).to_string())

# Lightweight pattern flags for error clustering
pat = pd.DataFrame(index=invalid_only.index)
pat['has_brgy_word'] = invalid_only['norm'].str.contains(r'\b(?:BRGY|BARANGAY|BGY)\b', regex=True, na=False)
pat['has_city_word'] = invalid_only['norm'].str.contains(r'\b(?:CITY|MUNICIPALITY|MUN)\b', regex=True, na=False)
pat['has_numbers'] = invalid_only['norm'].str.contains(r'\d', regex=True, na=False)
pat['short_addr'] = invalid_only['norm'].str.len() <= 28
pat['street_only_tokens'] = invalid_only['norm'].str.contains(
    r'\b(?:ST|STREET|AVE|AVENUE|ROAD|RD|BLK|LOT|UNIT|PHASE|B\d+|L\d+)\b', regex=True, na=False
)

cluster = pd.DataFrame({
    'missing_city_word': (~pat['has_city_word']).astype(int),
    'with_brgy_word': pat['has_brgy_word'].astype(int),
    'street_only_tokens': pat['street_only_tokens'].astype(int),
    'short_addr': pat['short_addr'].astype(int),
})
print('\nPattern prevalence among invalid rows:')
print(cluster.mean().mul(100).round(1).astype(str) + '%')

print('\nSample invalid addresses (first 30):')
display(invalid_only[['original_address', 'address_normalized', 'city_signal', 'city_name', 'province_name', 'invalid_reason']].head(30))

# Show likely false-invalid candidates (have city-like cues but still invalid)
candidates = invalid_only[
    invalid_only['norm'].str.contains(
        r'\b(?:PQUE|PARANAQUE|PARAÑAQUE|CEBU|QC|QUEZON|MANILA|MAKATI|TAGUIG)\b',
        regex=True, na=False
    )
]
print(f'\nPotentially recoverable invalids with city-like cues: {len(candidates):,}')
display(candidates[['original_address', 'address_normalized', 'city_signal', 'city_name', 'province_name', 'invalid_reason']].head(30))

Total rows: 224,121
Invalid rows: 78,388

Top invalid reasons:
invalid_reason
barangay_unresolved           47693
missing_explicit_city         29321
barangay_ambiguous_no_city     1361
province_city_conflict           13

Pattern prevalence among invalid rows:
missing_city_word     51.0%
with_brgy_word         8.3%
street_only_tokens    22.7%
short_addr            20.3%
dtype: str

Sample invalid addresses (first 30):


,original_address,address_normalized,city_signal,city_name,province_name,invalid_reason
143579,48B AQURIUS ST PAMPLONA DOS LASPINAS CITY,48B AQURIUS STREET PAMPLONA DOS LAS PI�AS CITY,PAMPLONA,PAMPLONA,CAGAYAN,barangay_unresolved
143580,3314 Apitong San Martin De Porres Pque,3314 APITONG SAN MARTIN DE PORRES PARA�AQUE CITY,NaN,NaN,NaN,missing_explicit_city
143581,475 malugay st san Martin de porres pque city,475 MALUGAY STREET SAN MARTIN DE PORRES PARA�A...,NaN,NaN,NaN,missing_explicit_city
143582,5225 San Lazaro St. San Antonio Valley5 Pque 1715,5225 SAN LAZARO STREET SAN ANTONIO VALLEY5 PAR...,SAN ANTONIO,SAN ANTONIO,NORTHERN SAMAR,barangay_unresolved
143583,370 malugay st. San martin de Porres pque,370 MALUGAY STREET SAN MARTIN DE PORRES PARA�A...,NaN,NaN,NaN,missing_explicit_city
143584,TIBIG LPA CITY,TIBIG LAS PINAS CITY,CITY OF LAS PIÑAS,CITY OF LAS PIÑAS,METRO MANILA,barangay_unresolved
143601,Las pinas,LAS PI�AS CITY,NaN,NaN,NaN,missing_explicit_city
143602,"115 CITTADELLA AVE, CITTADELLA EXECUTIVE VILLA...",115 CITTADELLA AVENUE CITTADELLA EXECUTIVE VIL...,NaN,NaN,NaN,missing_explicit_city
143603,831 Green Revolution Street CAA Laspinas City,831 GREEN REVOLUTION STREET CAA LAS PI�AS CITY,NaN,NaN,NaN,missing_explicit_city
143604,B INTERNATIONAL LASPINAS CITY,B INTERNATIONAL LAS PI�AS CITY,NaN,NaN,NaN,missing_explicit_city



Potentially recoverable invalids with city-like cues: 11,515


,original_address,address_normalized,city_signal,city_name,province_name,invalid_reason
144470,29 A Progress st manila times village laspinas...,29 A PROGRESS STREET MANILA TIMES VILLAGE LAS ...,NaN,NaN,NaN,missing_explicit_city
145485,845 Angalo St San Nicolas Manila,845 ANGALO STREET SAN NICOLAS MANILA,SAN NICOLAS,SAN NICOLAS,BATANGAS,barangay_unresolved
145491,279 Jaboneros St San Nicolas Manila,279 JABONEROS STREET SAN NICOLAS MANILA,SAN NICOLAS,SAN NICOLAS,BATANGAS,barangay_unresolved
145799,Manila Cityhall,MANILA CITYHALL,NaN,NaN,NaN,missing_explicit_city
145800,manila/CITYHALL,MANILA CITYHALL,NaN,NaN,NaN,missing_explicit_city
145801,MANILA AVIDA,MANILA AVIDA,NaN,NaN,NaN,missing_explicit_city
145802,Manila CITY,MANILA CITY,NaN,NaN,NaN,missing_explicit_city
145803,MANILA ICTY,MANILA ICTY,NaN,NaN,NaN,missing_explicit_city
145804,MANILA CITY HALL,MANILA CITY HALL,NaN,NaN,NaN,missing_explicit_city
145805,MANILA CITYHALL,MANILA CITYHALL,NaN,NaN,NaN,missing_explicit_city


In [20]:
# Cell 14 -- Recoverable invalids snapshot (plain text)
cand = out_df[
    (~out_df['is_valid']) &
    out_df['address_normalized'].fillna('').astype(str).str.contains(
        r'\b(?:PQUE|PARANAQUE|PARAÑAQUE|QC|QUEZON|MANILA|MKT|MAKATI|TAGUIG|PASIG|MARIKINA|CEBU|DAVAO|CDO|CAGAYAN DE ORO)\b',
        regex=True, na=False
    )
].copy()

print(f'Recoverable candidates: {len(cand):,}')
cols = ['original_address', 'address_normalized', 'city_signal', 'city_name', 'province_name', 'invalid_reason']
for i, row in cand[cols].head(60).iterrows():
    print('-' * 100)
    print(f"ORIG : {row['original_address']}")
    print(f"NORM : {row['address_normalized']}")
    print(f"SIG  : {row['city_signal']} | CITY: {row['city_name']} | PROV: {row['province_name']} | REASON: {row['invalid_reason']}")

print('\nCity-not-resolved rows:')
unres = out_df[out_df['invalid_reason'] == 'city_not_resolved'][cols]
for i, row in unres.head(60).iterrows():
    print('-' * 100)
    print(f"ORIG : {row['original_address']}")
    print(f"NORM : {row['address_normalized']}")
    print(f"SIG  : {row['city_signal']} | CITY: {row['city_name']} | PROV: {row['province_name']} | REASON: {row['invalid_reason']}")

Recoverable candidates: 16,985
----------------------------------------------------------------------------------------------------
ORIG : 29 A Progress st manila times village laspinas city
NORM : 29 A PROGRESS STREET MANILA TIMES VILLAGE LAS PI�AS CITY
SIG  : nan | CITY: nan | PROV: nan | REASON: missing_explicit_city
----------------------------------------------------------------------------------------------------
ORIG : 845 Angalo St San Nicolas Manila
NORM : 845 ANGALO STREET SAN NICOLAS MANILA
SIG  : SAN NICOLAS | CITY: SAN NICOLAS | PROV: BATANGAS | REASON: barangay_unresolved
----------------------------------------------------------------------------------------------------
ORIG : 279 Jaboneros St San Nicolas Manila
NORM : 279 JABONEROS STREET SAN NICOLAS MANILA
SIG  : SAN NICOLAS | CITY: SAN NICOLAS | PROV: BATANGAS | REASON: barangay_unresolved
----------------------------------------------------------------------------------------------------
ORIG : Manila Cityhall
NORM :

In [21]:
# Cell 15 -- Compact unresolved diagnostics
tmp = out_df.copy()
tmp['norm'] = tmp['address_normalized'].fillna('').astype(str)

unres = tmp[tmp['invalid_reason'] == 'city_not_resolved'].copy()
print(f"city_not_resolved rows: {len(unres):,}")
for s in unres['norm'].head(20).tolist():
    print(' -', s[:140])

# Count metro-manila style abbreviations still not resolved
abbr_patterns = {
    'PQUE': r'\bPQUE\b',
    'MKT': r'\bMKT\b',
    'QC': r'\bQC\b',
    'MNL': r'\bMNL\b',
    'BGC': r'\bBGC\b',
    'CDO': r'\bCDO\b',
}
print('\nUnresolved abbreviation counts:')
for k, pat in abbr_patterns.items():
    c = int((~tmp['is_valid'] & tmp['norm'].str.contains(pat, regex=True, na=False)).sum())
    print(f' {k}: {c}')

city_not_resolved rows: 0

Unresolved abbreviation counts:
 PQUE: 0
 MKT: 0
 QC: 0
 MNL: 0
 BGC: 3
 CDO: 0


In [22]:
# Cell 16 -- Unresolved detail view (plain text sample)
cols = ['original_address', 'address_normalized', 'city_signal', 'province_name', 'invalid_reason']
u = out_df[out_df['invalid_reason'] == 'city_not_resolved'][cols].copy()
print(f'city_not_resolved count: {len(u):,}')
for i, row in u.head(25).iterrows():
    print('-' * 90)
    print('ORIG :', str(row['original_address'])[:150])
    print('NORM :', str(row['address_normalized'])[:150])
    print('SIG  :', row['city_signal'], '| PROV:', row['province_name'], '| REASON:', row['invalid_reason'])

city_not_resolved count: 0


In [23]:
# Cell 17 -- Focus checks (La Carlota and Tondo)
tmp = out_df.copy()
tmp['norm'] = tmp['address_normalized'].fillna('').astype(str)

la = tmp[tmp['city_name'].fillna('').str.contains('LA CARLOTA', case=False, na=False)].copy()
la_false = la[~la['norm'].str.contains('LA CARLOTA', case=False, na=False)]
print(f'Rows with city_name=LA CARLOTA*: {len(la):,}')
print(f'LA CARLOTA assigned without mention: {len(la_false):,}')

tondo_rows = tmp[tmp['norm'].str.contains(r'\bTONDO\b', case=False, regex=True, na=False)].copy()
print(f'Rows mentioning TONDO: {len(tondo_rows):,}')
if len(tondo_rows):
    print('TONDO status counts:')
    print(tondo_rows['is_valid'].value_counts().to_string())

Rows with city_name=LA CARLOTA*: 61
LA CARLOTA assigned without mention: 2
Rows mentioning TONDO: 3,187
TONDO status counts:
is_valid
True     3052
False     135


In [24]:
# Cell 18 -- Manila district city-key check from dim_location city_name
manila_cores = [
    'TONDO', 'BINONDO', 'ERMITA', 'INTRAMUROS', 'MALATE', 'PACO',
    'PANDACAN', 'PORT AREA', 'QUIAPO', 'SAMPALOC', 'SAN NICOLAS',
    'SANTA ANA', 'SANTA CRUZ',
]

def _canon_tmp(s: str) -> str:
    return re.sub(r'\s+', ' ', re.sub(r'[^A-Z0-9]+', ' ', str(s).upper())).strip()

canon_cities = [_canon_tmp(c) for c in cities]
print('Has CITY OF MANILA:', 'CITY OF MANILA' in canon_cities)
print('Has MANILA:', 'MANILA' in canon_cities)

print('\nDistrict core coverage in city_name:')
for core in manila_cores:
    ccore = _canon_tmp(core)
    hits = sorted({cities[i] for i, c in enumerate(canon_cities) if f' {ccore} ' in f' {c} '})
    print(f'[{core}] -> {len(hits)} match(es)')
    for h in hits[:8]:
        print('  -', h)

Has CITY OF MANILA: False
Has MANILA: False

District core coverage in city_name:
[TONDO] -> 1 match(es)
  - TONDO I/II
[BINONDO] -> 1 match(es)
  - BINONDO
[ERMITA] -> 1 match(es)
  - ERMITA
[INTRAMUROS] -> 1 match(es)
  - INTRAMUROS
[MALATE] -> 1 match(es)
  - MALATE
[PACO] -> 1 match(es)
  - PACO
[PANDACAN] -> 1 match(es)
  - PANDACAN
[PORT AREA] -> 1 match(es)
  - PORT AREA
[QUIAPO] -> 1 match(es)
  - QUIAPO
[SAMPALOC] -> 1 match(es)
  - SAMPALOC
[SAN NICOLAS] -> 1 match(es)
  - SAN NICOLAS
[SANTA ANA] -> 1 match(es)
  - SANTA ANA
[SANTA CRUZ] -> 1 match(es)
  - SANTA CRUZ


In [25]:
# Cell 19 -- Tondo mapping probe in dim_location
probe = dim[dim['barangay_name'].fillna('').astype(str).str.contains('TONDO', case=False, na=False)].copy()
print(f'Rows with barangay containing TONDO: {len(probe):,}')
if len(probe):
    print('Unique city names:')
    for c in sorted(probe['city_name'].dropna().unique().tolist())[:20]:
        print(' -', c)
    print('Unique province names:')
    for p in sorted(probe['province_name'].dropna().unique().tolist())[:10]:
        print(' -', p)

Rows with barangay containing TONDO: 6
Unique city names:
 - ANDA
 - CABANGAN
 - CLAVER
 - SAN JOSE CITY
 - SAN JUAN
 - TANGALAN
Unique province names:
 - AKLAN
 - LA UNION
 - NUEVA ECIJA
 - PANGASINAN
 - SURIGAO DEL NORTE
 - ZAMBALES


In [26]:
# Targeted Manila district sanity check
for raw in ['TONDO MANILA', 'BINONDO MANILA', 'ERMITA MANILA', 'INTRAMUROS MANILA']:
    addr = normalize_alias(raw, compiled_re, replace_map)
    ans = match_address(addr, dim, cities, provinces, regions, barangays)
    print(raw, '=>', ans['city_signal'], '|', ans['city_name'], '|', ans['invalid_reason'])

TONDO MANILA => TONDO I/II | TONDO I/II | None
BINONDO MANILA => BINONDO | BINONDO | None
ERMITA MANILA => ERMITA | ERMITA | None
INTRAMUROS MANILA => INTRAMUROS | INTRAMUROS | None


# 🔧 IMPROVED PIPELINE (Confidence + Hierarchical Matching + Status)

This section contains an alternative address matching approach using legacy schema for barangay-focused matching.

It leverages fuzzy string matching and dim_location hierarchy to resolve addresses with confidence scoring. This pipeline:
- Normalizes addresses using the alias dictionary
- Extracts barangay and city candidates using regex patterns
- Performs hierarchical fuzzy matching with confidence scoring
- Outputs matched results to Excel with hierarchical validation status

In [27]:
# Compatibility bootstrap for the legacy "IMPROVED PIPELINE" cells below.
# It maps current notebook variables to the names expected by that section.

def normalize_text(text):
    import re, pandas as pd
    if pd.isna(text):
        return ''
    text = text.upper()
    text = re.sub(r'LANDMARK.*', '', text)
    text = re.sub(r'[^A-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Ensure df exists with Original_Address column.
if 'df' not in globals():
    if 'out_df' in globals() and isinstance(out_df, pd.DataFrame) and not out_df.empty:
        df = out_df.copy()
    elif 'df_input' in globals() and isinstance(df_input, pd.DataFrame) and not df_input.empty:
        df = df_input.copy()
    else:
        df = pd.read_excel(INPUT_FILE)

if 'Original_Address' not in df.columns:
    if 'original_address' in df.columns:
        df['Original_Address'] = df['original_address']
    else:
        df['Original_Address'] = df.iloc[:, 0].astype(str)

# Build legacy alias schema expected by downstream cells.
if 'alias' not in globals():
    if 'alias_df' in globals() and {'pattern', 'replacement'}.issubset(alias_df.columns):
        alias = alias_df[['pattern', 'replacement']].copy()
        alias = alias.rename(columns={'pattern': 'alias', 'replacement': 'standard_name'})
    else:
        alias = pd.DataFrame(columns=['alias', 'standard_name'])

# Build legacy dim_location schema expected by downstream cells.
if 'dim_location' not in globals():
    if 'dim' in globals() and {'barangay_name', 'city_name'}.issubset(dim.columns):
        dim_location = dim.copy()
        rename_map = {
            'barangay_name': 'barangay',
            'city_name': 'city',
            'province_name': 'province',
            'region_name': 'region',
        }
        dim_location = dim_location.rename(columns=rename_map)
    else:
        dim_location = pd.DataFrame(columns=['barangay', 'city', 'province', 'region'])

print(f'df rows: {len(df):,} | alias rows: {len(alias):,} | dim_location rows: {len(dim_location):,}')

df rows: 224,121 | alias rows: 512 | dim_location rows: 42,011


In [28]:

import re

def extract_components(text):
    result = {"barangay": None, "city": None}

    brgy_match = re.search(r'(BRGY|BARANGAY)?\s*([A-Z\-\s]{3,})', text)
    if brgy_match:
        result["barangay"] = brgy_match.group(2).strip()

    city_patterns = [
        r'QUEZON CITY', r'MANILA', r'BAGUIO', r'LAS\s*PINAS', r'DAVAO', r'CEBU'
    ]

    for pattern in city_patterns:
        if re.search(pattern, text):
            result["city"] = pattern.replace("\\s*", " ")

    return result


In [29]:

df['clean_address'] = df['Original_Address'].apply(normalize_text)

components = df['clean_address'].apply(extract_components)
df['brgy_candidate'] = components.apply(lambda x: x['barangay'])
df['city_candidate'] = components.apply(lambda x: x['city'])


In [30]:

alias_dict = dict(zip(alias['alias'].str.upper(), alias['standard_name'].str.upper()))

def apply_alias(val):
    if pd.isna(val):
        return None
    return alias_dict.get(val.upper(), val)

df['brgy_norm'] = df['brgy_candidate'].apply(apply_alias)
df['city_norm'] = df['city_candidate'].apply(apply_alias)


In [31]:

from rapidfuzz import fuzz, process

barangay_list = dim_location['barangay'].dropna().unique()

def match_address(row):
    import pandas as pd

    brgy = row['brgy_norm']
    city = row['city_norm']

    score = 0
    match = None

    if pd.notna(brgy):
        res = process.extractOne(brgy, barangay_list, scorer=fuzz.ratio)
        if res:
            match_name, s, _ = res
            if s >= 85:
                score += 40
                match = match_name

    candidates = dim_location.copy()

    if match is not None:
        candidates = candidates[candidates['barangay'] == match]

    if pd.notna(city):
        candidates = candidates[candidates['city'].str.contains(city, na=False)]
        score += 40

    if len(candidates) > 1:
        return pd.Series([None, "AMBIGUOUS", score])

    if len(candidates) == 1 and score >= 70:
        return pd.Series([candidates.iloc[0]['barangay'], "MATCHED", score])

    return pd.Series([None, "LOW_CONFIDENCE", score])


In [32]:

df[['matched_barangay', 'MATCH_STATUS', 'CONFIDENCE_SCORE']] = df.apply(match_address, axis=1)


KeyboardInterrupt: 

In [ ]:

df.to_excel('cleaned_output_v2.xlsx', index=False)
print("✅ Improved pipeline complete!")


✅ Improved pipeline complete!
